In [ ]:
%%html
<!-- CSS settings for this notebook -->
<style>
    h1 {color:purple}
    h2 {color:purple}
    h3 {color:#0099ff}
    hr {    
        border: 0;
        height: 3px;
        background: #333;
        background-image: linear-gradient(to right, ForestGreen, DeepSkyBlue, ForestGreen);
    }
</style>

# Data Mining with PubNub

---

# Objectives
* **Data-mine** a **live social-media stream** using the **PubNub** real-time messaging platform
* **Clean and preprocess** social media post text to prepare it for analysis
* **Translate non-English posts** to English with the **DeepL AI Translator**
* **Stream posts** and display them in real time using a **`SubscribeCallback`** subclass
* Perform **sentiment analysis** on a live social-media-post stream with **TextBlob**
* **Geocode** post sender locations with **geopy** and the free **ArcGIS** service, then **map them worldwide** with **folium**

---

# 12.1 Introduction
* **Data mining** &mdash; searching large collections of data for **insights**
* Social-media posts reveal how people **feel** about products, events and issues
* **Sentiment** in posts can help **make predictions**:
    * **Stock prices**
    * **Election results**
    * Likely **revenues** for a **new movie** or **product**
    * **Success** of a company's **marketing campaign**
* Spot **comments** about your company's products and competitors' products
* Spot **trending topics** as they emerge
* Connect to live data streams with **easy-to-use web services**

## What Is Data Mining?
* **Data mining** applies techniques from statistics, machine learning and databases to **extract patterns and knowledge** from large datasets
* Common data-mining tasks include:
    * **Classification** &mdash; assign items to predefined categories
    * **Clustering** &mdash; group similar items together without predefined categories
    * **Association rule mining** &mdash; find items that frequently appear together
    * **Sentiment analysis** &mdash; determine the emotional tone of text
    * **Trend detection** &mdash; identify topics that are gaining popularity

---

# 12.2 The PubNub Sample Twitter Stream
* **PubNub** is a real-time data streaming platform used in production apps worldwide
    > https://www.pubnub.com
* PubNub provides **free sample data channels** for learning and prototyping
* Up to 1,000,000 free transactions per month
* `pubnub` Python module for managing streams
* The **`pubnub-twitter`** channel streams a **live sample of real tweets** from around the world
    * No Twitter developer account required &mdash; free and open
    * Subscribe key: `sub-c-d00e0d32-66ac-4628-aa65-a42a1f0c493b`

<hr style="height:2px; border:none; color:#000; background-color:#000;">

## Post Message Format
Each message arriving on `pubnub-twitter` is a **Twitter API v1.1 tweet object** — a Python dict with fields including:

| Field | Description |
| :--- | :--- |
| `text` | Tweet text (up to 280 chars) |
| `user` | Dict containing `screen_name`, `name`, `location`, etc. |
| `lang` | Twitter-detected language code (e.g., `'en'`, `'es'`, `'ja'`, `'und'` for undetermined) |
| `coordinates` | Exact GPS point or `None` |
| `place` | Tagged place object or `None` |
| `created_at` | UTC timestamp string |

```json
{
  "text": "Amazing view! #travel https://t.co/abc123",
  "user": {
    "screen_name": "traveler99",
    "location": "London, UK"
  },
  "lang": "en"
}
```

---

# 12.3 Libraries Used in This Notebook

## PubNub Python SDK
* https://github.com/pubnub/python
> `pip install pubnub`

## tweet-preprocessor
* https://github.com/s/preprocessor
* Cleans post text before NLP analysis (removes URLs, @mentions, reserved words, etc.)
> `pip install tweet-preprocessor`

## better-profanity
* https://github.com/snguyenthanh/better_profanity
* Censors profanity with `****`
> `pip install better-profanity`

## DeepL AI Translator
* https://github.com/DeepLcom/deepl-python
* Translates text between 30+ languages with high accuracy
> `pip install --upgrade deepl`
* Requires a **free API key** — allows 500,000 characters/month:
    1. Go to https://www.deepl.com/pro#developer
    2. Click **API** → **Sign up for free**
    3. Under **DeepL API Free** click **Sign up for free**
    4. Provide an email and credit card (to prevent abuse; free tier is not charged)
    5. After registration, click **Account Management** → **Account** tab
    6. Scroll to **Authentication Key for DeepL API** and copy your key
    7. In `keys.py`, replace the empty string in `deepL_key = ''` with your key

## TextBlob
* https://textblob.readthedocs.io/
* Simple NLP library including sentiment analysis
> `pip install textblob`

## geopy
* https://github.com/geopy/geopy
* Converts location strings (e.g., `'Boston, MA'`) into latitudes and longitudes
* We use the **free ArcGIS geocoding service** &mdash; no API key required
> `conda install -c conda-forge geopy`

## pandas
> `pip install pandas`

## folium
* https://github.com/python-visualization/folium
* Creates interactive maps using the **Leaflet.js** JavaScript library and **OpenStreetMap** tiles
> `pip install folium`

**Maps from OpenStreetMap.org**
* Map tiles are copyrighted by the OpenStreetMap contributors
* www.openstreetmap.org/copyright &nbsp;&nbsp; www.opendatacommons.org/licenses/odbl

---

# 12.4 Configuring Keys and PubNub
* Store your credentials in `keys.py`:
    * **`pubnub_user_id`** &mdash; any unique non-empty string (e.g., your email)
    * **`deepL_key`** &mdash; your DeepL API key (see installation instructions above)
* **`PNConfiguration`** holds the connection settings for a PubNub client
* **`subscribe_key`** &mdash; the public key for the PubNub sample twitter stream
* **`PubNub(config)`** creates the client object used to subscribe to channels

In [ ]:
import keys

In [ ]:
from pubnub.pnconfiguration import PNConfiguration

config = PNConfiguration()

# key for simulated social media stream -- normally not embedded in your code
config.subscribe_key = 'sub-c-d00e0d32-66ac-4628-aa65-a42a1f0c493b' 
config.user_id = keys.pubnub_user_id

* We'll reuse `config` across all three streaming examples in this notebook

---

# 12.5 Cleaning/Preprocessing Posts
* **Data cleaning** is one of a data scientist's most important — and most time-consuming — tasks
* Raw post text often contains **noise** that can mislead NLP tools:
    * **`@mentions`** — references to other accounts (e.g., `@nasa`)
    * **URLs** — `https://t.co/...` shortened links that carry no sentiment
    * **Reserved words** — `RT` (retweet prefix) and `FAV`
    * **Emoji** and **smileys**
    * **Hashtags** — useful as metadata, but `#` symbols can confuse tokenizers
* Normalizing text before analysis is called **preprocessing** and it measurably improves NLP accuracy

### **tweet-preprocessor** Library
* https://github.com/s/preprocessor
* Call **`p.set_options`** once to specify what to remove, then **`p.clean(text)`** to apply

| Option | Removes |
| :--- | :--- |
| **`OPT.MENTION`** | `@`-mentions (e.g., `@nasa`) |
| **`OPT.EMOJI`** | Emoji |
| **`OPT.HASHTAG`** | Hashtags (e.g., `#mars`) |
| **`OPT.NUMBER`** | Numbers |
| **`OPT.RESERVED`** | Reserved words &mdash; `RT` and `FAV` |
| **`OPT.SMILEY`** | Smileys |
| **`OPT.URL`** | URLs |

### **better-profanity** Library
* Call `profanity.load_censor_words()` once at startup, then `profanity.censor(text)` replaces profanity with `****`
* Applied to **all English text** — both native English posts and translated non-English posts

## **Example:** Cleaning a Post

### Start with a raw post containing a retweet marker, an `@`-mention, emoji and a URL

In [ ]:
post_text = 'RT @NASA Check this out! 🚀 Just landed on Mars https://t.co/exampleLink #Space #Exploration'
post_text

### Import **tweet-preprocessor** and configure it to remove `RT`, `@`-mentions, URLs and emoji

In [ ]:
import preprocessor as p

In [ ]:
p.set_options(p.OPT.RESERVED, p.OPT.MENTION, p.OPT.URL, p.OPT.EMOJI)
cleaned = p.clean(post_text)
cleaned

### Censor any profanity

In [ ]:
from better_profanity import profanity
profanity.load_censor_words()
profanity.censor(cleaned)

* After cleaning, the text is ready for NLP tasks such as sentiment analysis

## **Example:** Translating a Non-English Post with DeepL

### Import DeepL and create a Translator object

In [ ]:
import deepl
translator = deepl.Translator(keys.deepL_key)

### Translate a Spanish post to English
* `target_lang='EN-US'` requests American English
* DeepL **auto-detects** the source language — no language code needed

In [ ]:
spanish_post = 'El tiempo hoy en Madrid es increíble. ☀️ #Madrid #España'
result = translator.translate_text(spanish_post, target_lang='EN-US')
print(f'ORIGINAL:   {spanish_post}')
print(f'TRANSLATED: {profanity.censor(result.text)}')

* The `translate_post` utility function in `postutilities.py` wraps this in a try/except so any translation failure falls back gracefully to the original text
* See the utility function discussion at the end of this notebook

---

# 12.6 **Example:** Streaming Posts
* Subscribe to `pubnub-twitter` and display each incoming post in real time
* **English posts** are cleaned and censored before display
* **Non-English posts** are displayed in their **original language** followed by a cleaned, censored **English translation**

## Class `PostListener` (in `postlistener.py`)
* Checks `post['lang']` to determine whether translation is needed
* If `lang` does **not** start with `'en'`:
    * Prints `ORIGINAL:` — the raw post text
    * Calls `postutilities.translate_post` 
    * Prints `TRANSLATED:` — the English result
* If `lang` starts with `'en'`: prints the censored text directly
* Sets a **`threading.Event`** (`done`) when `POST_LIMIT` is reached

## Setting Up PubNub and the Listener

In [ ]:
from pubnub.pubnub import PubNub
import postlistener

post_listener = postlistener.PostListener(limit=10)
pubnub = PubNub(config)
pubnub.add_listener(post_listener)

## Starting the Stream

In [ ]:
pubnub.subscribe().channels('pubnub-twitter').execute()
post_listener.done.wait()  # block until POST_LIMIT is reached
print(f'\nReceived {post_listener.post_count} posts.')

---

# 12.7 **Example:** Sentiment Analysis
* **Sentiment analysis** &mdash; determine the **emotional tone** of text
    * **Positive**: upbeat, enthusiastic, favorable
    * **Negative**: critical, frustrated, unfavorable
    * **Neutral**: factual, mixed, no strong opinion
* The `pubnub-twitter` stream is **multilingual** — posts arrive in dozens of languages
* Non-English posts are **translated to English before scoring** so TextBlob's English-trained dictionary gives consistent, meaningful results
* Applications: product monitoring, financial modeling, election forecasting

## Class `SentimentListener` (in `sentimentlistener.py`)
* Skips retweets (text starting with `'RT'`)
* If non-English, translates with `postutilities.translate_post` before scoring
* Scores polarity with **TextBlob** on the English text:
    * polarity > 0.1 → **positive** (`+`)
    * polarity < &minus;0.1 → **negative** (`−`)
    * otherwise → **neutral** (` `)
* For non-English posts, prints both `ORIGINAL` and `TRANSLATED` lines
* Sets `done` event when `POST_LIMIT` is reached

## **Example:** Analyzing Sentiment in a Live Post Stream

In [ ]:
limit = 10

In [ ]:
sentiment_dict = {'positive': 0, 'neutral': 0, 'negative': 0}

In [ ]:
import sentimentlistener

sentiment_listener = sentimentlistener.SentimentListener(
    sentiment_dict=sentiment_dict, limit=limit)

pubnub2 = PubNub(config)
pubnub2.add_listener(sentiment_listener)
pubnub2.subscribe().channels('pubnub-twitter').execute()
sentiment_listener.done.wait()
print(f'Post sentiment for {limit} posts:')
print(f'Positive: {sentiment_dict["positive"]}')
print(f' Neutral: {sentiment_dict["neutral"]}')
print(f'Negative: {sentiment_dict["negative"]}')

---

# 12.8 **Example:** Geocoding and Mapping Post Locations
* Collect streaming posts, geocode sender locations, then plot them on an interactive world map
* Twitter users can optionally add a **location string** to their profile (e.g., `'London, UK'` or `'São Paulo'`)
* **Geocoding** converts a location string into a **latitude** and **longitude** for plotting
* We use the **free ArcGIS geocoding service** via **geopy** — no API key required
* Non-English post text is **translated to English** so all map popups are readable

## **geopy** Library
* https://github.com/geopy/geopy
* Supports dozens of geocoding web services, many with free tiers
* We use the **ArcGIS** geocoder — free, no API key required

```python
from geopy import ArcGIS
geo = ArcGIS()
location = geo.geocode('London, UK')
print(location.latitude, location.longitude)  # 51.5074, -0.1278
```

## Class `LocationListener` (in `locationlistener.py`)
* Reads `post['user']['location']` — the sender's profile location string
* Skips posts whose sender has no location set
* Translates non-English post text to English with `postutilities.translate_post` so map popups are readable
* Stores each located post as `{username, text, location}` — `latitude` and `longitude` added later by `get_geocodes()`
* Sets `done` event when `POST_LIMIT` located posts have been collected

## Collecting Posts with User Location Strings

In [ ]:
posts = []
counts = {'total_posts': 0, 'locations': 0}

In [ ]:
import locationlistener

location_listener = locationlistener.LocationListener(
    counts_dict=counts, posts_list=posts, limit=50)

pubnub3 = PubNub(config)
pubnub3.add_listener(location_listener)
pubnub3.subscribe().channels('pubnub-twitter').execute()
location_listener.done.wait()
print(f'\nCollected {counts["locations"]} located posts out of {counts["total_posts"]} total.')

In [ ]:
print(f'Posts with a user location: {counts["locations"] / counts["total_posts"]:.1%}')

## Geocoding the Locations

* `get_geocodes` converts each `post['location']` string to a latitude and longitude using **ArcGIS**
* Adds `'latitude'` and `'longitude'` keys to each post dict in-place
* Returns the count of location strings ArcGIS could not resolve

In [ ]:
from postutilities import get_geocodes
bad_locations = get_geocodes(posts)
print(f'Ungeocoded locations: {bad_locations}')

## Building the World Map with Folium

In [ ]:
import pandas as pd

df = pd.DataFrame(posts)
df

In [ ]:
df = df.dropna()  # remove rows where geocoding failed
df

In [ ]:
import folium

world_map = folium.Map(location=[20, 0], zoom_start=2, detect_retina=True)

In [ ]:
for t in df.itertuples():
    popup_text = f'<b>{t.username}</b> &mdash; {t.location}<br>{t.text}'
    popup = folium.Popup(popup_text, max_width=300)
    folium.Marker(
        location=(t.latitude, t.longitude),
        popup=popup,
        tooltip=t.location
    ).add_to(world_map)

In [ ]:
world_map.save('post_map.html')
world_map

---

## Utility Functions in `postutilities.py`

### `translate_post(text)`
Translates post text to English using **DeepL**. DeepL auto-detects the source language, so no language code is needed. Falls back to returning the original text if translation fails for any reason.

```python
def translate_post(text):
    try:
        return translator.translate_text(text, target_lang='EN-US')
    except Exception:
        return text  # graceful fallback
```

### `get_geocodes(post_list)`
Iterates over a list of post dicts and geocodes each `post['location']` string using the **free ArcGIS service** from **geopy**. Adds `'latitude'` and `'longitude'` keys in-place and returns a count of strings that could not be resolved.

```python
def get_geocodes(post_list):
    geo = ArcGIS()
    for post in post_list:
        geo_location = geo.geocode(post['location'])
        if geo_location:
            post['latitude']  = geo_location.latitude
            post['longitude'] = geo_location.longitude
    ...
```

---

&copy;1992&ndash;2025 by Pearson Education, Inc. All Rights Reserved. This content is based on Chapter 12 of the book [**Intro to Python for Computer Science and Data Science: Learning to Program with AI, Big Data and the Cloud**](https://amzn.to/2VvdnxE).

DISCLAIMER: The authors and publisher of this book have used their best efforts in preparing the book. These efforts include the development, research, and testing of the theories and programs to determine their effectiveness. The authors and publisher make no warranty of any kind, expressed or implied, with regard to these programs or to the documentation contained in these books. The authors and publisher shall not be liable in any event for incidental or consequential damages in connection with, or arising out of, the furnishing, performance, or use of these programs.